In [2]:
!pip install rasterio numpy

In [3]:
import rasterio 
import numpy as np

In [4]:
# Dictionary mapping feature names to their GeoTIFF file paths

files = {
    "openness": "sample3openess_pos.tif",  # positive openness RVT feature
    "slope": "sample3slope.tif",     # slope RVT feature
    "SVF": "Sample3svf.tif"     # SVF RVT feature
}

In [5]:
Sample_fraction = 1.0 # 1.0 use for all pixels

In [6]:
results = {}  
low_pnct , high_pnct = 2, 98 # percentile range to clip outliers

In [7]:
for name, path in files.items():
    print(f"\n Reading {name} ({path})")
    with rasterio.open(path) as ds:
        arr = ds.read(1, masked = True) #  hide nodata pixels
        data = arr.compressed() # Drop nodata pixel values
    # sample subset if fraction < 1.0
        if Sample_fraction <1.0:
            n = int(len(data)*Sample_fraction)  # calculate how many pixel to keep 
            indx = np.random.choice(len(data),size = n , replace = False) # randomly pick n pixels
            data = data[indx]
    # calculate 2nd and 98th percentile for clippping outliers
        pnct_low , pnct_high = np.percentile(data, [low_pnct , high_pnct])
        mean = data.mean()
        std = data.std()

    # print result summary
        results[name] = (pnct_low , pnct_high)
        print(f" min = {data.min():.4f} max = {data.max():.4f}")
        print(f" mean = {mean:.4f} std = {std:.4f}")
        print( f" {low_pnct}th pnct = {pnct_low:.4f} {high_pnct}th pnct = {pnct_high :.4f}" )


 Reading openness (sample3openess_pos.tif)
 min = 7.9206 max = 159.2492
 mean = 87.3565 std = 3.3663
 2th pnct = 77.4913 98th pnct = 91.2449

 Reading slope (sample3slope.tif)
 min = 0.0000 max = 89.1976
 mean = 9.5725 std = 8.3586
 2th pnct = 0.7164 98th pnct = 36.1389

 Reading SVF (Sample3svf.tif)
 min = 0.0156 max = 1.0000
 mean = 0.9230 std = 0.0639
 2th pnct = 0.7210 98th pnct = 0.9931


In [8]:
# Fixed stretch values to use across all tiles
for name , (pnct_low , pnct_high) in results.items():
    print(f"{name}: low = {pnct_low:.4f} , high = {pnct_high:.4f}")
    

    

openness: low = 77.4913 , high = 91.2449
slope: low = 0.7164 , high = 36.1389
SVF: low = 0.7210 , high = 0.9931
